# 음성·음악 혼합 학습 실험
기존 음악 전용 판별기와 새 혼합 학습 판별기를 비교합니다. 독립 최종 검증이나 제출용이 아닙니다.
실제/생성 음성의 문장과 생성기는 학습·개발에서 분리했지만 기준 화자는 같습니다. 음악은 기존 학습·개발 자료입니다.
T4 GPU를 선택하고 모두 실행한 뒤 mixed_probe_bundle.zip을 업로드하세요. 결과 mixed_probe_results.zip을 보내주세요.


In [ ]:
import torch, sys, platform
assert torch.cuda.is_available(), "런타임 → 런타임 유형 변경 → GPU를 선택하세요."
print(torch.cuda.get_device_name(0), torch.__version__, sys.version)
print("무료 GPU 환경의 진단입니다. L4 제출 서버 재현 검증은 별도입니다.")

## 1. mixed_probe_bundle.zip 업로드


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, zipfile, tempfile, json, shutil, os
uploaded = files.upload()
assert len(uploaded) == 1, "mixed_probe_bundle.zip 하나만 선택하세요."
blob = next(iter(uploaded.values()))
assert hashlib.sha256(blob).hexdigest() == "fad4f51c51480b7ab4aac30ab22b94441729adeb3ea2ea224a33a7079ee76fef", "다른 버전의 ZIP입니다. 새 실험 ZIP을 선택하세요."
WORK = Path(tempfile.mkdtemp(prefix="deepvoice_probe_", dir="/content"))
import io
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for name in z.namelist():
        assert (WORK / name).resolve().is_relative_to(WORK.resolve())
    z.extractall(WORK)
RESULTS = WORK / "results"
RESULTS.mkdir()
manifest = json.loads((WORK / "probe/manifest.json").read_text())
for row in manifest["rows"]:
    path = WORK / "probe/test" / (row["ID"] + ".wav")
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row["clip_sha256"]
print("학습 실험 음원 무결성 확인 완료", WORK)

## 2. 이전 실행에서 사용한 패키지 설치 / 버전 기록

In [ ]:
# 설치 실패를 숨기지 않고 전체 로그를 저장한다. 서버와 환경이 완전히 같지는 않다.
import subprocess, sys, pathlib
installed = subprocess.run([sys.executable, "-m", "pip", "install",
    "demucs==4.0.1", "panns-inference==0.1.1", "librosa==0.10.2.post1", "soundfile", "soxr"],
    capture_output=True, text=True)
pathlib.Path("/content/install.log").write_text(installed.stdout + installed.stderr)
print((installed.stdout + installed.stderr)[-6000:])
installed.check_returncode()
import importlib
for mod in ["librosa", "demucs", "panns_inference", "transformers", "torchaudio"]:
    m = importlib.import_module(mod)
    print(mod, getattr(m, "__version__", "?"))
(RESULTS / "environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True) + "\n" + sys.version + "\n" + torch.cuda.get_device_name(0))

## 3. 모델 준비 — 이전 Colab 파일이 남아 있으면 재사용

In [ ]:
import pathlib, hashlib, shutil, os
MODEL = WORK / "model"
def pilot_file_sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(4 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()
expected = dict(line.split() for line in (MODEL / "SHA256SUMS.txt").read_text().splitlines() if line.strip())
# 위 파일은 digest -> relative path 순서다.
for checksum, relative in expected.items():
    previous = Path("/content/submit/model") / relative
    target = MODEL / relative
    if not target.exists() and previous.is_file() and pilot_file_sha256(previous) == checksum:
        target.parent.mkdir(parents=True, exist_ok=True)
        try:
            os.link(previous, target)
        except OSError:
            shutil.copy2(previous, target)
have_all = all((MODEL / rel).is_file() and pilot_file_sha256(MODEL / rel) == checksum for checksum, rel in expected.items())

if not have_all:
    import hashlib, pathlib, shutil, os
    from huggingface_hub import snapshot_download

    # WORK는 이번 실험 전용 폴더
    MODEL = WORK / "model"

    def sha256(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 22), b""):
                h.update(chunk)
        return h.hexdigest()

    expected = {}
    for line in (MODEL / "SHA256SUMS.txt").read_text().splitlines():
        if line.strip():
            digest, rel = line.split()
            expected[rel] = digest

    # --- DF-Arena 1B (대회 배포본과 동일 리비전) ---
    snap = snapshot_download(repo_id="Speech-Arena-2025/DF_Arena_1B_V_1", revision="fb6ce85de12c2c5a509d89114adaf827dd75f49f",
                             allow_patterns=["pytorch_model.bin"])
    shutil.copy2(pathlib.Path(snap) / "pytorch_model.bin", MODEL / "df_arena_1b" / "pytorch_model.bin")

    # --- HTDemucs (demucs 가 캐시에 받아둔 .th 를 로컬 repo 로 옮긴다) ---
    from demucs.pretrained import get_model as _dl
    import torch
    _orig = torch.load
    torch.load = lambda *a, **k: _orig(*a, **{**k, "weights_only": False})
    try:
        _dl("htdemucs")
    finally:
        torch.load = _orig
    cache = pathlib.Path(torch.hub.get_dir()) / "checkpoints"
    for th in cache.glob("*.th"):
        shutil.copy2(th, MODEL / "htdemucs" / th.name)

    # --- PANNs Cnn14 (panns_inference 가 ~/panns_data 에 받는다) ---
    from panns_inference import AudioTagging
    AudioTagging(checkpoint_path=None, device="cpu")
    shutil.copy2(pathlib.Path.home() / "panns_data" / "Cnn14_mAP=0.431.pth",
                 MODEL / "panns" / "Cnn14_mAP=0.431.pth")

    print()
    ok = True
    for rel, digest in expected.items():
        path = MODEL / rel
        if not path.exists():
            print(f"[FAIL] 없음 {rel}"); ok = False; continue
        got = sha256(path)
        mark = "OK  " if got == digest else "FAIL"
        if got != digest:
            ok = False
        print(f"[{mark}] {rel}  {got[:16]}...")
    print("\n무결성", "일치 — 대회 배포본과 같은 가중치다" if ok else "불일치! 이 검증은 신뢰할 수 없다")
for line in (MODEL / "SHA256SUMS.txt").read_text().splitlines():
    if line.strip():
        checksum, relative = line.split()
        assert pilot_file_sha256(MODEL / relative) == checksum, relative
print("모델 해시 검사 통과")

## 혼합 특징 추출 → 학습 → 개발 비교
기존 임베딩 위의 작은 판별기만 학습합니다. C 세 후보를 개발 조건별 평균 EER로 선택합니다.
홀드아웃·대회 데이터는 사용하지 않습니다. 전처리·학습 코드와 특징·예측·환경을 결과 ZIP에 보관합니다.


In [ ]:
import subprocess, sys, shutil, pathlib
worker_results = WORK / "mixed_probe_results"
worker_results.mkdir(exist_ok=True)
shutil.copy2(RESULTS / "environment.txt", worker_results / "environment.txt")
log_path = worker_results / "run.log"
try:
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen([sys.executable, "-u", "run_mixed_probe.py", "--work", str(WORK)], cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            returncode = process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
    assert returncode == 0, "실험이 중단됐습니다. 결과 ZIP 또는 출력 노트북을 보내주세요."
finally:
    if pathlib.Path("/content/install.log").exists():
        shutil.copy2("/content/install.log", worker_results / "install.log")
    archive = shutil.make_archive("/content/mixed_probe_results", "zip", worker_results)
    files.download(archive)
